# FPL Weekly Assistant

Every week: update **FREE_TRANSFERS**, add any injury news you know that the model
can't read, then run the report.

The engine lives in one file that this notebook and the backtest both import, so
the settings you tune in the backtest are the settings that run here.


In [1]:
# Setup - fetches the engine. Run once per session.
import os, sys, subprocess

# The one copy of the engine. Point this at your repo once you've pushed it;
# if fpl_engine.py is already sitting next to this notebook it is used as-is,
# so local edits work without pushing anything.
ENGINE_URL = "https://raw.githubusercontent.com/PKnott/fpl/main/fpl_engine.py"

if not os.path.exists("fpl_engine.py"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pulp", "requests"])
    import requests
    r = requests.get(ENGINE_URL, timeout=60); r.raise_for_status()
    open("fpl_engine.py", "w").write(r.text)
    print("engine downloaded")

import fpl_engine as E
print("engine ready")


engine ready


## Your settings

Edit these. Everything else - the model, the planner, the chip rules - lives in
`E.DEFAULTS` and is what the backtest tunes; you shouldn't need to touch it.


In [2]:
E.USER.update(
    TEAM_ID=7399216,        # the number in your "Points" page URL
    FREE_TRANSFERS=1,       # the one thing FPL doesn't publish - update weekly

    # Injury news the model can't read: {player: {gameweek: chance 0-1}}
    AVAILABILITY_OVERRIDES={},
    # How often you expect a player to start (0-1), e.g. a new signing:
    START_OVERRIDES={},
    LOCK=[],                # never sell
    EXCLUDE=[],             # never buy
)

cfg = E.resolve_cfg()
print(f"planning {cfg['PLAN_WEEKS']} weeks, long view {cfg['LONG_VIEW']}, "
      f"chip thresholds {cfg['CHIP_TOP_PCT']}")


planning 5 weeks, long view 19, chip thresholds {'3xc': 0.15, 'bboost': 0.4, 'freehit': 0.05, 'wildcard': 0.15}


## The weekly report

In [3]:
report = E.run(cfg)

Loading player histories for 182 players...


 FPL ASSISTANT - GW6   plan GW6-10, long view to GW24, projections to GW38
Bank £2.0m | Free transfers 1 | Chips: Bench Boost, Triple Captain

YOUR SQUAD
Player                    Sell  Next   12wk  Fixture / price outlook
GK Verbruggen              4.5   3.5   17.9  SUN (A)
GK Woodman                 4.0   0.0    0.0  MCI (H)
DEF Calafiori              5.7   5.0   23.1  LEE (H) ↑ likely rise
DEF De Cuyper              4.7   3.9   22.2  SUN (A)
DEF Gvardiol               5.7   3.5   20.6  LIV (A) ↑ likely rise
DEF Konsa                  4.5   2.8   16.0  LEE (H)  ⚠ Unspecified injury - 75% chance of playing
DEF Mendy                  4.0   1.2    5.7  EVE (H)
MID Tavernier              6.1   4.7   22.2  CHE (A)
MID Groß                   5.7   4.0   21.3  SUN (A)
MID Rogers                 7.6   4.2   20.3  BOU (H) ↑ likely rise
MID Ødegaard               6.7   4.2   19.5  LEE (H)
MID Gakpo                  7.2   2.6   16.2  MCI (H)  ⚠ Ankle injury - 75% chance of playing
FWD Haaland  


 CHIPS  (compared across every week of each chip's window)
  Bench Boost: +8.6 now, this week #11 of 14 remaining (top 79%, playing if in top 43%) -> save - best remaining week is GW19 (+16.8)
  Triple Captain: +5.7 now, this week #13 of 14 remaining (top 93%, playing if in top 21%) -> save - best remaining week is GW7 (+7.9)

 TRANSFERS
Options this week (each is the best multi-week plan starting that way):

 0 transfer(s): +0.0 pts vs rolling

 1 transfer(s): +3.2 pts vs rolling
   OUT Gakpo (LIV, £7.2m)               IN Mbeumo (MUN, £7.9m)

 2 transfer(s)  (-6 hit): +1.7 pts vs rolling
   OUT Konsa (ARS, £4.6m)               IN Thomas (COV, £4.0m)
   OUT Gakpo (LIV, £7.2m)               IN Mbeumo (MUN, £7.9m)

 3 transfer(s)  (-12 hit): -0.5 pts vs rolling
   OUT Konsa (ARS, £4.6m)               IN Thomas (COV, £4.0m)
   OUT Gakpo (LIV, £7.2m)               IN Saka (ARS, £9.5m)
   OUT João Pedro (CHE, £7.7m)          IN Barry (EVE, £5.7m)

 4 transfer(s)  (-18 hit): -3.0 pts vs rol

## Why is a player projected what he is?

Add any names - yours or not. Ambiguous names take a club: `"Gakpo|LIV"`.


In [4]:
report.explain("Haaland", "Saka")


POINTS BREAKDOWN - GW6
                                   Haaland            Saka
Fixture                            LIV (A)         LEE (H)
Starts this season                  5 of 5          5 of 5
Last 5 games (newest 1st)                -  80,90,89,90,67
Chance of starting                    100%            100%
Chance of 60+ minutes                  92%            100%
Expected minutes                        90              84
Availability                          100%            100%
----------------------------------------------------------
Playing time                         +1.92           +2.00
Goals                                +1.97           +2.02
Assists                              +0.18           +0.71
Clean sheet                          +0.00           +0.49
Def. contributions                   +0.00           +0.07
Bonus                                +0.70           +0.57
Yellow cards                         -0.12           -0.07
--------------------------------

## Team defence: the rating next to what actually happened

In [5]:
report.team_table()


TEAM DEFENCE (league avg goals per team per game: 1.51)
Team   Games  Conceded  xG agst  Pre-season  Keeper busy  Adjust   RATING
              per game per game      rating     shots/90
CRY        5      2.20     1.89        108%          6.7    1.23     142%
IPS        5      2.20     1.80        115%          6.2    1.19     141%
CHE        5      2.40     1.78         85%          7.0    1.26     126%
NEW        5      1.80     1.83        100%          5.6    1.13     122%
FUL        5      1.60     1.82        108%          4.8    1.04     117%
AVL        5      1.80     1.84        100%          5.0    1.07     115%
COV        5      2.00     1.73        115%          3.8    0.93     107%
HUL        5      0.80     1.48        115%          4.2    0.98     103%
TOT        5      1.60     1.56        108%          4.0    0.95     101%
LEE        5      0.60     1.29        100%          4.6    1.02      94%
MUN        5      1.60     1.36         85%          4.8    1.04      92

## Chip detail

Every remaining week of each chip's window, so you can see what the ranking is
built on rather than taking the verdict on trust.


In [6]:
import pandas as pd
rows = {E.CHIP_NAMES[c]: v["values"] for c, v in report.chips.items()}
if rows:
    display(pd.DataFrame(rows).rename_axis("GW").round(1))
else:
    print("No chips available.")


,Bench Boost,Triple Captain
GW,,
6,8.6,5.6
7,7.5,7.9
8,8.4,6.0
9,8.4,5.7
10,12.2,6.2
11,12.2,6.9
12,14.7,5.6
13,14.3,6.0
14,14.4,5.9
